## PydanticAI 제품 비교 리뷰 분석

실전_1_1에서 PydanticAI로 구조화한 리뷰 분석 결과를 활용하여, **3개 선크림 제품** 의 비교 분석을 수행합니다.

| 학습 목표 | 내용 |
|---|---|
| 구조화된 출력의 가치 | AI가 생성한 정형 데이터를 Python으로 자유롭게 가공 |
| 제품 비교 통계 | 항목별 긍정/부정 비율, 감정 분포, 장단점 집계 |
| 시각화 | seaborn/matplotlib로 비교 차트 생성 |
| AI 요약 | 비교 분석 결과를 PydanticAI로 한줄 요약 |

**선수 학습**: 실전_1_1 (리뷰 데이터 전처리)

### 환경 설정

In [ ]:
# ========================================
# 필요한 라이브러리를 불러옵니다
# ========================================

import os
import json
import platform
from collections import defaultdict, Counter
from itertools import combinations
from typing import List, Literal

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
from scipy.stats import chi2_contingency
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from pydantic_ai import Agent
from pydantic_ai.models.google import GoogleModelSettings

# 운영체제별 한글 폰트 설정
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
    korean_font_path = 'C:/Windows/Fonts/malgun.ttf'
elif platform.system() == 'Darwin':  # macOS
    plt.rcParams['font.family'] = 'AppleGothic'
    korean_font_path = '/System/Library/Fonts/AppleSDGothicNeo.ttc'
else:  # Linux
    plt.rcParams['font.family'] = 'NanumGothic'
    korean_font_path = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'

plt.rcParams['axes.unicode_minus'] = False

# .env 파일에서 API 키 로드
load_dotenv()
gemini_model = os.getenv('GEMINI_MODEL', 'gemini-3.1-flash-lite-preview')
model_id = f'google-gla:{gemini_model}'

# 비교 분석 결과 저장 폴더
output_dir = 'outputs'
os.makedirs(output_dir, exist_ok=True)

print(f"모델: {model_id}")
print(f"출력 폴더: {output_dir}/")

### 1. 분석 결과 로드

**실습 (1)**
- 실전_1_1에서 저장한 `product_review_analysis.json` 파일을 로드합니다.
- 구조화된 출력 덕분에 JSON을 DataFrame으로 바로 변환하여 분석할 수 있습니다.

In [ ]:
# =============================================================================
#  분석 결과 불러오기
# =============================================================================

analysis_file = os.path.join(output_dir, 'product_review_analysis.json')

if not os.path.exists(analysis_file):
    print(f"[오류] {analysis_file} 파일이 없습니다.")
    print("실전_1_1_리뷰데이터_전처리.ipynb 를 먼저 실행해주세요.")
else:
    analysis_df = pd.read_json(analysis_file, orient='records')

    print(f"분석 결과 로드 완료: {len(analysis_df)}개 리뷰")
    print(f"컬럼: {list(analysis_df.columns)}")
    print()

    # 제품별 기본 통계
    print("제품별 리뷰 수:")
    for pid, group in analysis_df.groupby('product_id'):
        name = group.iloc[0]['product_name'][:40]
        print(f"  [{pid}] {name}... ({len(group)}개)")

    print()
    display(analysis_df[['product_name', 'rating', 'overall_sentiment', 'pros', 'cons', 'recommendation_score']].head())

### 2. 제품별 비교 통계

**실습 (2)**
- 제품별로 평점, 추천도, 감정 분포, 항목별 긍정/부정 비율, 장단점을 집계합니다.
- 한국어 `Literal` 값을 사용했으므로 영문 => 한글 매핑이 필요 없습니다.

In [ ]:
# =============================================================================
#  제품별 비교 통계 계산
# =============================================================================

print("제품 비교 분석 시작...")
print()

# 워드클라우드용 불용어
stopwords = {'없음', '있음', '사용', '좋음', '나쁨', '보통', '괜찮음', '추천', '만족', '불만족', '상품'}

# 제품별 통계 저장
product_stats = {}

for product_id, group in analysis_df.groupby('product_id'):
    product_name = group.iloc[0]['product_name']

    stats = {
        'name': product_name,
        'review_count': len(group),
        'avg_rating': group['rating'].mean(),
        'rating_std': group['rating'].std() if len(group) > 1 else 0,
        'avg_recommendation': group['recommendation_score'].mean(),
        'sentiment_distribution': group['overall_sentiment'].value_counts().to_dict()
    }

    aspect_sentiment = defaultdict(list)
    aspect_positive = defaultdict(int)
    aspect_negative = defaultdict(int)
    aspect_keywords_pos = defaultdict(list)
    aspect_keywords_neg = defaultdict(list)
    all_pros = []
    all_cons = []

    for _, review in group.iterrows():
        if isinstance(review['pros'], list):
            filtered = [p for p in review['pros'] if p.strip() not in stopwords]
            all_pros.extend(filtered)
        if isinstance(review['cons'], list):
            filtered = [c for c in review['cons'] if c.strip() not in stopwords]
            all_cons.extend(filtered)

        # 항목별 감정 수집 (이미 한국어 값)
        if isinstance(review['aspects'], list):
            for aspect in review['aspects']:
                if isinstance(aspect, dict):
                    aspect_name = aspect.get('aspect')
                    sentiment = aspect.get('sentiment')
                    keywords = aspect.get('keywords', [])
                    if aspect_name and sentiment:
                        aspect_sentiment[aspect_name].append(sentiment)
                        if sentiment == '긍정':
                            aspect_positive[aspect_name] += 1
                            aspect_keywords_pos[aspect_name].extend(keywords)
                        elif sentiment == '부정':
                            aspect_negative[aspect_name] += 1
                            aspect_keywords_neg[aspect_name].extend(keywords)

    aspect_scores = {}
    for asp_name, sentiments in aspect_sentiment.items():
        pos_count = sum(1 for s in sentiments if s == '긍정')
        if len(sentiments) > 0:
            aspect_scores[asp_name] = pos_count / len(sentiments)

    stats['aspect_scores'] = aspect_scores
    stats['aspect_positive'] = dict(aspect_positive)
    stats['aspect_negative'] = dict(aspect_negative)
    stats['aspect_keywords_pos'] = dict(aspect_keywords_pos)
    stats['aspect_keywords_neg'] = dict(aspect_keywords_neg)
    stats['top_pros'] = [item for item, _ in Counter(all_pros).most_common(5)]
    stats['top_cons'] = [item for item, _ in Counter(all_cons).most_common(5)]
    stats['all_pros'] = all_pros
    stats['all_cons'] = all_cons

    product_stats[product_id] = stats

for pid, stats in product_stats.items():
    print(f"[{pid}] {stats['name'][:40]}")
    print(f"  리뷰: {stats['review_count']}개 | 평균 평점: {stats['avg_rating']:.2f} | 추천도: {stats['avg_recommendation']:.2f}")
    print(f"  장점: {', '.join(stats['top_pros'][:3])}")
    print(f"  단점: {', '.join(stats['top_cons'][:3])}")
    print()

In [ ]:
# =============================================================================
#  비교 요약 텍스트 생성 및 저장
# =============================================================================

summary_lines = []
summary_lines.append("제품 비교 요약:")
summary_lines.append("=" * 60)

for pid, stats in product_stats.items():
    summary_lines.append(f"\n제품: {stats['name'][:40]}")
    summary_lines.append(f"  리뷰 수: {stats['review_count']}개")
    summary_lines.append(f"  평균 평점: {stats['avg_rating']:.2f}점")
    summary_lines.append(f"  평균 추천도: {stats['avg_recommendation']:.2f}")
    summary_lines.append(f"  주요 장점: {', '.join(stats['top_pros'][:5])}")
    summary_lines.append(f"  주요 단점: {', '.join(stats['top_cons'][:5])}")

    if stats['aspect_scores']:
        summary_lines.append(f"  측면별 만족도 (긍정 비율):")
        sorted_aspects = sorted(stats['aspect_scores'].items(), key=lambda x: x[1], reverse=True)
        for aspect_name, score in sorted_aspects:
            summary_lines.append(f"    - {aspect_name}: {score:.2f} ({score*100:.1f}%)")

product_comparison_summary = "\n".join(summary_lines)
print(product_comparison_summary)

# 텍스트 파일로 저장
summary_path = f"{output_dir}/product_comparison_summary.txt"
with open(summary_path, 'w', encoding='utf-8') as f:
    f.write(product_comparison_summary)
print(f"\n저장 완료: {summary_path}")

### 3. 시각화

**실습 (3)**
- 제품별 평점/추천도 비교, 항목별 긍정 비율 히트맵, 감정 분포, 워드클라우드를 생성합니다.
- 모든 차트는 `analysis_outputs/` 폴더에 PNG로 저장됩니다.

In [ ]:
# =============================================================================
#  1. 평균 평점 및 추천도 비교
# =============================================================================

products = list(product_stats.keys())
product_names = [product_stats[pid]['name'][:20] + '...' for pid in products]

avg_ratings = [product_stats[pid]['avg_rating'] for pid in products]
rating_stds = [product_stats[pid]['rating_std'] for pid in products]
recommendations = [product_stats[pid]['avg_recommendation'] for pid in products]

fig, ax1 = plt.subplots(figsize=(10, 6))

x = np.arange(len(products))
bars = ax1.bar(x, avg_ratings, width=0.4, color='#5B9BD5', label='평균 평점', yerr=rating_stds, capsize=5)
ax1.set_ylabel('평점 (1~5점)', color='#5B9BD5')
ax1.set_ylim(0, 5.5)

for bar, val in zip(bars, avg_ratings):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.15, f'{val:.2f}',
             ha='center', va='bottom', fontweight='bold', color='#5B9BD5')

ax2 = ax1.twinx()
ax2.plot(x, recommendations, 'o-', color='#ED7D31', linewidth=2, markersize=8, label='추천도')
for i, val in enumerate(recommendations):
    ax2.text(i, val + 0.03, f'{val:.2f}', ha='center', fontweight='bold', color='#ED7D31')
ax2.set_ylabel('추천도 (0~1)', color='#ED7D31')
ax2.set_ylim(0, 1.1)

ax1.set_xticks(x)
ax1.set_xticklabels(product_names, rotation=15, ha='right')
ax1.set_title('제품별 평균 평점 및 추천도 비교', fontsize=14, fontweight='bold')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

plt.tight_layout()
plt.savefig(f"{output_dir}/rating_recommendation_comparison.png", dpi=150, bbox_inches='tight', pad_inches=0.2)
plt.show()

In [ ]:
# =============================================================================
#  2. 항목별 긍정 비율 히트맵
# =============================================================================

all_aspects = set()
for stats in product_stats.values():
    all_aspects.update(stats['aspect_scores'].keys())
all_aspects = sorted(list(all_aspects))

heatmap_data = []
for pid in products:
    row = [product_stats[pid]['aspect_scores'].get(asp, np.nan) for asp in all_aspects]
    heatmap_data.append(row)

heatmap_df = pd.DataFrame(heatmap_data, index=product_names, columns=all_aspects)

fig, ax = plt.subplots(figsize=(12, 5))
sns.heatmap(
    heatmap_df, annot=True, fmt='.2f', cmap='RdYlGn',
    vmin=0, vmax=1, center=0.5, linewidths=0.5,
    cbar_kws={'label': '긍정 비율'}, ax=ax
)
ax.set_title('제품별 항목 긍정 비율 히트맵', fontsize=14, fontweight='bold')
ax.set_ylabel('')

plt.tight_layout()
plt.savefig(f"{output_dir}/aspect_positive_heatmap.png", dpi=150, bbox_inches='tight', pad_inches=0.2)
plt.show()

In [ ]:
# =============================================================================
#  3. 감정 분포 비교
# =============================================================================

sentiment_order = ['긍정', '혼합', '중립', '부정']
sentiment_colors = {'긍정': '#4CAF50', '혼합': '#FFC107', '중립': '#9E9E9E', '부정': '#F44336'}

pid_to_short = {pid: product_stats[pid]['name'][:20] + '...' for pid in products}
analysis_df['product_short'] = analysis_df['product_id'].map(pid_to_short)

sentiment_vals = [s for s in sentiment_order if s in analysis_df['overall_sentiment'].values]

fig, ax = plt.subplots(figsize=(10, 6))
sns.countplot(
    data=analysis_df, x='product_short', hue='overall_sentiment',
    hue_order=sentiment_vals,
    palette={s: sentiment_colors[s] for s in sentiment_vals},
    ax=ax
)
ax.set_title('제품별 감정 분포 비교', fontsize=14, fontweight='bold')
ax.set_xlabel('')
ax.set_ylabel('건수')
ax.legend(title='감정')

plt.tight_layout()
plt.savefig(f"{output_dir}/sentiment_distribution_comparison.png", dpi=150, bbox_inches='tight', pad_inches=0.2)
plt.show()

In [ ]:
# =============================================================================
#  4. 제품별 장단점 워드클라우드
# =============================================================================

for pid, stats in product_stats.items():
    product_name = stats['name'][:30]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    pros_text = ' '.join(stats['all_pros']) if stats['all_pros'] else '데이터없음'
    wc_pros = WordCloud(
        font_path=korean_font_path, width=600, height=400,
        background_color='white', colormap='Greens',
        stopwords=stopwords, max_words=50
    ).generate(pros_text)
    axes[0].imshow(wc_pros, interpolation='bilinear')
    axes[0].set_title('장점', fontsize=12, fontweight='bold', color='#4CAF50')
    axes[0].axis('off')

    cons_text = ' '.join(stats['all_cons']) if stats['all_cons'] else '데이터없음'
    wc_cons = WordCloud(
        font_path=korean_font_path, width=600, height=400,
        background_color='white', colormap='Reds',
        stopwords=stopwords, max_words=50
    ).generate(cons_text)
    axes[1].imshow(wc_cons, interpolation='bilinear')
    axes[1].set_title('단점', fontsize=12, fontweight='bold', color='#F44336')
    axes[1].axis('off')

    plt.suptitle(f'{product_name} - 장단점 워드클라우드', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f"{output_dir}/wordcloud_{pid}.png", dpi=150, bbox_inches='tight', pad_inches=0.2)
    plt.show()

In [ ]:
# =============================================================================
#  5. 제품별 항목별 긍정/부정 키워드 워드클라우드
# =============================================================================

for pid, stats in product_stats.items():
    product_name = stats['name'][:30]

    aspects_with_data = []
    for asp in stats['aspect_keywords_pos'].keys() | stats['aspect_keywords_neg'].keys():
        if stats['aspect_keywords_pos'].get(asp, []) or stats['aspect_keywords_neg'].get(asp, []):
            aspects_with_data.append(asp)

    if not aspects_with_data:
        continue

    n_aspects = len(aspects_with_data)
    fig, axes = plt.subplots(n_aspects, 2, figsize=(14, 3.5 * n_aspects))
    if n_aspects == 1:
        axes = axes.reshape(1, -1)

    for idx, asp in enumerate(aspects_with_data):
        pos_text = ' '.join(stats['aspect_keywords_pos'].get(asp, [])) or '데이터없음'
        wc = WordCloud(
            font_path=korean_font_path, width=400, height=200,
            background_color='white', colormap='Greens', max_words=30
        ).generate(pos_text)
        axes[idx, 0].imshow(wc, interpolation='bilinear')
        axes[idx, 0].set_title(f'{asp} - 긍정', fontsize=10, color='#4CAF50', pad=12)
        axes[idx, 0].axis('off')

        neg_text = ' '.join(stats['aspect_keywords_neg'].get(asp, [])) or '데이터없음'
        wc = WordCloud(
            font_path=korean_font_path, width=400, height=200,
            background_color='white', colormap='Reds', max_words=30
        ).generate(neg_text)
        axes[idx, 1].imshow(wc, interpolation='bilinear')
        axes[idx, 1].set_title(f'{asp} - 부정', fontsize=10, color='#F44336', pad=12)
        axes[idx, 1].axis('off')

    plt.suptitle(f'{product_name} - 항목별 키워드', fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.subplots_adjust(hspace=0.4)
    plt.savefig(f"{output_dir}/aspect_keywords_{pid}.png", dpi=150, bbox_inches='tight', pad_inches=0.3)
    plt.show()

In [ ]:
# =============================================================================
#  6. 제품별 레이더 차트: 항목별 긍정/부정 비율
# =============================================================================
import numpy as np

# 전체 제품에서 공통 항목 추출 (최소 3건 이상 데이터가 있는 항목만)
MIN_COUNT = 3
all_aspect_names = sorted(set(
    asp for stats in product_stats.values()
    for asp in stats['aspect_scores'].keys()
))

# 각 항목의 전체 언급 수 확인 -> 너무 적은 항목 제외
common_aspects = []
for asp in all_aspect_names:
    total = sum(
        stats['aspect_positive'].get(asp, 0) + stats['aspect_negative'].get(asp, 0)
        for stats in product_stats.values()
    )
    if total >= MIN_COUNT:
        common_aspects.append(asp)


n_products = len(product_stats)
fig, axes = plt.subplots(1, n_products, figsize=(7 * n_products, 7),
                            subplot_kw=dict(polar=True))
if n_products == 1:
    axes = [axes]

# 레이더 차트 각도 계산
angles = np.linspace(0, 2 * np.pi, n_aspects, endpoint=False).tolist()
angles += angles[:1]

for idx, (pid, stats) in enumerate(product_stats.items()):
    ax = axes[idx]
    product_name = stats['name'][:25]

    # 항목별 긍정/부정 비율 계산
    pos_ratio = []
    neg_ratio = []
    for asp in common_aspects:
        pos = stats['aspect_positive'].get(asp, 0)
        neg = stats['aspect_negative'].get(asp, 0)
        total = pos + neg
        if total > 0:
            pos_ratio.append(pos / total)
            neg_ratio.append(neg / total)
        else:
            pos_ratio.append(0)
            neg_ratio.append(0)

    pos_closed = pos_ratio + [pos_ratio[0]]
    neg_closed = neg_ratio + [neg_ratio[0]]

    # 긍정 비율 (초록)
    ax.fill(angles, pos_closed, alpha=0.2, color='#4CAF50')
    ax.plot(angles, pos_closed, color='#4CAF50', linewidth=2, label='긍정 비율')
    ax.scatter(angles[:-1], pos_ratio, color='#4CAF50', s=40, zorder=5)

    # 부정 비율 (빨강)
    ax.fill(angles, neg_closed, alpha=0.2, color='#F44336')
    ax.plot(angles, neg_closed, color='#F44336', linewidth=2, label='부정 비율')
    ax.scatter(angles[:-1], neg_ratio, color='#F44336', s=40, zorder=5)

    # 축 설정
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(common_aspects, fontsize=10)
    ax.set_ylim(0, 1)
    ax.set_yticks([0.25, 0.5, 0.75, 1.0])
    ax.set_yticklabels(['25%', '50%', '75%', '100%'], fontsize=8, color='grey')
    ax.set_title(product_name, fontsize=13, fontweight='bold', pad=20)
    ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=9)

plt.suptitle('제품별 항목 긍정/부정 비율 (레이더 차트)', fontsize=15, fontweight='bold', y=1.03)
plt.tight_layout()
plt.savefig(f"{output_dir}/product_radar_charts.png", dpi=150, bbox_inches='tight', pad_inches=0.3)
plt.show()


### 4. 측면 간 상관관계 분석

**실습 (4)**
- 제품별로 항목 간 피어슨 상관계수를 계산하고 히트맵으로 시각화합니다.
- 동시에 긍정/부정으로 평가되는 항목 쌍을 파악합니다.

In [ ]:
# =============================================================================
#  측면 간 상관관계 분석
# =============================================================================

sentiment_score_map = {'긍정': 1, '중립': 0, '부정': -1}

for pid, stats in product_stats.items():
    product_name = stats['name'][:40]
    product_df = analysis_df[analysis_df['product_id'] == pid]

    print(f"{'=' * 60}")
    print(f"제품: {product_name}")
    print(f"{'=' * 60}")

    aspect_scores_list = []
    for _, review in product_df.iterrows():
        review_aspects = {}
        if isinstance(review['aspects'], list):
            for asp in review['aspects']:
                if isinstance(asp, dict):
                    name = asp.get('aspect')
                    sent = asp.get('sentiment')
                    if name and sent:
                        review_aspects[name] = sentiment_score_map.get(sent, 0)
        if review_aspects:
            aspect_scores_list.append(review_aspects)

    aspect_scores_df = pd.DataFrame(aspect_scores_list).fillna(0)

    if aspect_scores_df.empty or len(aspect_scores_df.columns) < 2:
        print("분석할 항목 데이터가 충분하지 않습니다.")
        print()
        continue

    corr_matrix = aspect_scores_df.corr(method='pearson')

    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(
        corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
        center=0, vmin=-1, vmax=1, square=True,
        cbar_kws={'label': '상관계수'}, ax=ax
    )
    ax.set_title(f'{product_name}\n항목 간 상관관계', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f"{output_dir}/aspect_correlation_{pid}.png", dpi=150, bbox_inches='tight', pad_inches=0.2)
    plt.show()

    print("\n강한 상관관계 (|r| >= 0.3):")
    found = False
    for i in range(len(corr_matrix.columns)):
        for j in range(i + 1, len(corr_matrix.columns)):
            val = corr_matrix.iloc[i, j]
            if abs(val) >= 0.3:
                a1 = corr_matrix.columns[i]
                a2 = corr_matrix.columns[j]
                relation = '정적 상관' if val > 0 else '부적 상관'
                print(f"  {a1} <=> {a2}: {val:.3f} ({relation})")
                found = True
    if not found:
        print("  해당 없음")
    print()

### 5. AI 비교 요약 생성

**실습 (5)**
- 비교 분석 통계 텍스트를 PydanticAI Agent에 입력하여, **구조화된 비교 요약** 을 생성합니다.
- 이것이 구조화된 출력의 핵심 가치입니다: AI가 생성한 정형 데이터를 코드로 가공한 뒤, 다시 AI에 입력하여 새로운 인사이트를 얻습니다.

In [ ]:
# =============================================================================
#  AI 비교 요약 생성
# =============================================================================

class ProductProfile(BaseModel):
    """개별 제품 프로필"""
    product_name: str = Field(description="제품명")
    strengths: List[str] = Field(description="주요 강점 (2~3개)")
    weaknesses: List[str] = Field(description="주요 약점 (1~2개)")
    best_for: str = Field(
        description="이 제품이 가장 적합한 사용자/상황",
        max_length=100
    )

class ComparisonSummary(BaseModel):
    """제품 비교 요약"""
    profiles: List[ProductProfile] = Field(
        description="각 제품의 강약점 프로필 (제품 수만큼)"
    )
    key_differences: List[str] = Field(
        description="제품 간 핵심 차이점 (2~3개)",
    )
    overall_recommendation: str = Field(
        description="상황별 추천 가이드 (예: 민감 피부라면 A, 가성비라면 B)",
        max_length=300
    )
    one_line_summary: str = Field(
        description="전체 비교 결과 한줄 요약",
        max_length=200
    )


summary_agent = Agent(
    model_id,
    output_type=ComparisonSummary,
    system_prompt=(
        "당신은 제품 비교 분석 전문가입니다. "
        "제공된 통계 데이터를 기반으로 각 제품의 강약점을 균형 있게 분석하고, "
        "순위보다는 제품별 특성과 상황별 추천에 초점을 맞춰 요약하세요."
    ),
)

prompt = f"""다음 제품 비교 분석 결과를 바탕으로 종합 요약을 작성해주세요.
각 제품의 강점과 약점을 균형 있게 분석하고, 어떤 사용자에게 어떤 제품이 적합한지 안내해주세요.

{product_comparison_summary}"""

result = await summary_agent.run(prompt, model_settings=GoogleModelSettings(temperature=0.3))
comparison = result.output

print("=== AI 비교 요약 ===")
print()
for p in comparison.profiles:
    print(f"■ {p.product_name}")
    print(f"  강점: {', '.join(p.strengths)}")
    print(f"  약점: {', '.join(p.weaknesses)}")
    print(f"  추천: {p.best_for}")
    print()
print(f"핵심 차이점:")
for diff in comparison.key_differences:
    print(f"  - {diff}")
print()
print(f"상황별 추천: {comparison.overall_recommendation}")
print(f"한줄 요약: {comparison.one_line_summary}")

### 결과 저장

생성된 파일들을 확인합니다. 이 파일들은 다음 노트북(실전_1_3)에서 종합 인사이트 도출의 입력 데이터로 사용됩니다.

In [ ]:
# =============================================================================
#  저장된 파일 확인
# =============================================================================

import glob

saved_files = sorted(glob.glob(f"{output_dir}/*"))
print(f"analysis_outputs/ 폴더에 저장된 파일 ({len(saved_files)}개):")
for f in saved_files:
    size_kb = os.path.getsize(f) / 1024
    print(f"  {os.path.basename(f)} ({size_kb:.1f} KB)")

print()
print("다음 단계: 실전_1_3_종합_인사이트도출.ipynb 에서 이 파일들을 로드하여 종합 인사이트를 도출합니다.")